# Machine Learning for Spatial Data — B
## Spatially-aware ML: does Random Forest match kriging?

In module A a plain Random Forest saw coordinates only as two more columns. That
is weak: a tree splitting on `x`/`y` draws box boundaries, not smooth spatial
structure — the "salt-and-pepper" problem. **Kriging** (module 02b) instead uses
the *variogram* — how similarity decays with distance — and is the geostatistics
gold standard for interpolation.

**RFsp** (Hengl et al. 2018, *PeerJ* — https://peerj.com/articles/5518/ ) closes
the gap with one idea: give the Random Forest the **buffer distances** from each
point to every observation as features. The forest can then learn the same
distance-decay structure the variogram encodes. Hengl's headline: *RFsp predicts
as accurately as ordinary kriging*.

**Data:** the **Meuse** heavy-metals dataset (`data/meuse/meuse.csv`) — the
canonical geostatistics teaching set (R `sp`/`gstat`), already used by
`02b_variograms`. Target: `log(zinc)`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import gstools as gs
from scipy.spatial.distance import cdist
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

RNG = 42
d = pd.read_csv("data/meuse/meuse.csv")
coords = d[["x", "y"]].values.astype(float)
z = np.log(d["zinc"].values.astype(float))            # target: log zinc
cov = d[["dist", "elev", "om"]].fillna(d[["dist", "elev", "om"]].median()).values
print("n =", len(d), " covariates: dist, elev, om")

n = 155  covariates: dist, elev, om


## Experiment 1 — interpolation from location only (fair vs kriging)

Kriging predicts from **location alone**. To compare fairly we give the forests
only location too, under the **same 5-fold CV** (refit each fold, Hengl's
protocol):

1. **Ordinary kriging (OK)** — variogram fit on the training fold.
2. **Plain RF** — features = raw coordinates `(x, y)`.
3. **RFsp** — features = **buffer distances to every training point**.

Buffer-distance trick: per fold, training features are the pairwise distance
matrix among training points; test features are distances from each test point to
those same training points (identical columns, so the forest transfers).

In [2]:
def rmse(a, b):
    return mean_squared_error(a, b) ** 0.5

def krige_fold(tr, te):
    bc, gamma = gs.vario_estimate((coords[tr, 0], coords[tr, 1]), z[tr])
    m = gs.Exponential(dim=2)
    m.fit_variogram(bc, gamma, nugget=True)
    ok = gs.krige.Ordinary(m, (coords[tr, 0], coords[tr, 1]), z[tr])
    pred, _ = ok((coords[te, 0], coords[te, 1]))
    return rmse(z[te], pred)

def rf_fold(tr, te, Xtr, Xte):
    rf = RandomForestRegressor(n_estimators=500, random_state=RNG, n_jobs=2)
    rf.fit(Xtr, z[tr])
    return rmse(z[te], rf.predict(Xte))

def rf_xy(tr, te):
    return rf_fold(tr, te, coords[tr], coords[te])

def rfsp(tr, te, extra=None):
    Xtr, Xte = cdist(coords[tr], coords[tr]), cdist(coords[te], coords[tr])
    if extra is not None:
        Xtr = np.hstack([Xtr, extra[tr]])
        Xte = np.hstack([Xte, extra[te]])
    return rf_fold(tr, te, Xtr, Xte)

kf = KFold(n_splits=5, shuffle=True, random_state=RNG)
res = {"OK (kriging)": [], "plain RF (x, y)": [], "RFsp (buffer dist)": []}
for tr, te in kf.split(coords):
    res["OK (kriging)"].append(krige_fold(tr, te))
    res["plain RF (x, y)"].append(rf_xy(tr, te))
    res["RFsp (buffer dist)"].append(rfsp(tr, te))

for k, v in res.items():
    print(f"{k:20s} CV-RMSE(log zinc) = {np.mean(v):.4f}  +/- {np.std(v):.4f}")

OK (kriging)         CV-RMSE(log zinc) = 0.4239  +/- 0.0789
plain RF (x, y)      CV-RMSE(log zinc) = 0.5019  +/- 0.1033
RFsp (buffer dist)   CV-RMSE(log zinc) = 0.4572  +/- 0.1074


**Reading it:** the coordinate-only Random Forest is worst — trees split space
into boxes (the salt-and-pepper artefact). Feeding the forest **buffer distances**
(RFsp) recovers most of the gap toward kriging, the gold standard — Hengl et al.'s
(2018) core result. On this tiny, smooth 155-point set kriging still edges ahead
(the paper's own caveat: small samples with near-linear trend favour model-based
kriging); RFsp's advantage grows with non-linearity and covariates — next.

## Experiment 2 — where ML pulls ahead: adding covariates

Real ML strength is folding **location *and* covariates** into one model. Meuse
zinc depends on distance-to-river and elevation. Kriging uses only location; RF
and RFsp can use both. Same 5-fold CV.

In [3]:
res2 = {"OK (location only)": res["OK (kriging)"],
        "RF (x, y + covars)": [], "RFsp (+ covars)": []}
for tr, te in kf.split(coords):
    res2["RF (x, y + covars)"].append(
        rf_fold(tr, te, np.hstack([coords[tr], cov[tr]]),
                np.hstack([coords[te], cov[te]])))
    res2["RFsp (+ covars)"].append(rfsp(tr, te, extra=cov))

for k, v in res2.items():
    print(f"{k:20s} CV-RMSE(log zinc) = {np.mean(v):.4f}  +/- {np.std(v):.4f}")

OK (location only)   CV-RMSE(log zinc) = 0.4239  +/- 0.0789
RF (x, y + covars)   CV-RMSE(log zinc) = 0.2992  +/- 0.0305
RFsp (+ covars)      CV-RMSE(log zinc) = 0.3026  +/- 0.0582


**Lesson —** with covariates, both forests now **beat** ordinary kriging —
because kriging cannot use `dist`/`elev` without extra machinery (regression
kriging). That is the practical case for spatially-aware ML: one model, spatial
structure *and* covariates.

**Wider toolbox:** gradient boosting (XGBoost / LightGBM) plugs in the same way,
with the spatial-lag / coordinate / Moran-eigenvector features from
`03b_features_engineering`. Kopczewska (2022, *Ann. Reg. Sci.*) frames this move
as ML *complementing* spatial econometrics, not replacing it. All of it stays
honest only under the spatial CV of module A.